# TN3K Dataset Verification

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skimage import io

RAW_DATA_DIR = Path("/project/data/ultrabench/tn3k/datasets/")

In [ ]:
# Check the contents of `calibration.json`
with open(RAW_DATA_DIR / "calibration.json") as f:
    calibration_data = json.load(f)

print(calibration_data.keys())
for key in calibration_data.keys():
    print(f"{key}: {calibration_data[key][:3]}")

In [ ]:
# Check the contents of `tn3k/label4test.csv` and `tn3k/label4trainval.csv`
columns = ["image", "label"]
trainval_df = pd.read_csv(RAW_DATA_DIR / "tn3k/label4trainval.csv", names=columns)
test_df = pd.read_csv(RAW_DATA_DIR / "tn3k/label4test.csv", names=columns)

display(trainval_df.head())
display(test_df.head())

In [ ]:
# Check that there are images and masks for each entry in the trainval and test
# CSV files
for _, row in trainval_df.iterrows():
    image_path = RAW_DATA_DIR / "tn3k/trainval-image" / row["image"]
    mask_path = RAW_DATA_DIR / "tn3k/trainval-mask" / row["image"]
    assert image_path.exists(), f"Image {image_path} does not exist"
    assert mask_path.exists(), f"Mask {mask_path} does not exist"

for _, row in test_df.iterrows():
    image_path = RAW_DATA_DIR / "tn3k/test-image" / row["image"]
    mask_path = RAW_DATA_DIR / "tn3k/test-mask" / row["image"]
    assert image_path.exists(), f"Image {image_path} does not exist"
    assert mask_path.exists(), f"Mask {mask_path} does not exist"

In [ ]:
# Load the images from the `tn3k/trainval-mask` and `tn3k/test-mask`
# directories using scikit-image and report:
# scikit-image and report:
#   - number of images
#   - number of images with N channels
#   - min/max pixel values
#   - min/mean/std/max height and width
image_paths = list((RAW_DATA_DIR / "tn3k/trainval-image").glob("*.jpg")) + list(
    (RAW_DATA_DIR / "tn3k/test-image").glob("*.jpg")
)
channel_counts = {}
min_pixel_value = float("inf")
max_pixel_value = float("-inf")
heights = []
widths = []
for image_path in image_paths:
    image = io.imread(image_path)
    channels = image.shape[2] if len(image.shape) == 3 else 1
    channel_counts[channels] = channel_counts.get(channels, 0) + 1
    min_pixel_value = min(min_pixel_value, image.min())
    max_pixel_value = max(max_pixel_value, image.max())
    heights.append(image.shape[0])
    widths.append(image.shape[1])

print(f"Number of images: {len(image_paths)}")
print(f"Number of images with N channels: {channel_counts}")
print(f"Min pixel value: {min_pixel_value}")
print(f"Max pixel value: {max_pixel_value}")
print(
    "Height: mean={:.2f}, std={:.2f} (min={}, max={})".format(
        np.mean(heights), np.std(heights), min(heights), max(heights)
    )
)
print(
    "Width: mean={:.2f}, std={:.2f} (min={}, max={})".format(
        np.mean(widths), np.std(widths), min(widths), max(widths)
    )
)

In [ ]:
# Load the masks from the `tn3k/trainval-mask` and `tn3k/test-mask`
# directories using scikit-image and report:
#   - number of masks with N channels
#   - unique pixel values
mask_files = list((RAW_DATA_DIR / "tn3k/trainval-mask").glob("*.jpg")) + list(
    (RAW_DATA_DIR / "tn3k/test-mask").glob("*.jpg")
)

channels = set()
pixel_values = set()
for mask_file in mask_files:
    mask = io.imread(mask_file)
    channels.add(mask.shape[2] if len(mask.shape) == 3 else 1)
    pixel_values.update(np.unique(mask).tolist())

print("Channels:", channels)
print("Unique pixel values:", pixel_values)

In [ ]:
# Display an image and its mask side by side
image = io.imread(RAW_DATA_DIR / "tn3k/trainval-image" / trainval_df.iloc[0]["image"])
mask = io.imread(RAW_DATA_DIR / "tn3k/trainval-mask" / trainval_df.iloc[0]["image"])

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image)
axes[0].set_title("Image")
axes[0].axis("off")
axes[1].imshow(mask)
axes[1].set_title("Mask")
axes[1].axis("off")
plt.show()